<a href="https://colab.research.google.com/github/joaoalexandre14/ex4_pml/blob/main/ex4pml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

# 1. Configure the device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Prepare the Data (Resize and convert 1 channel to 3 channels)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# 3. Load the pre-trained model and adjust the final layer
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze the initial layers to speed up the process (Focused fine-tuning)
for param in model.parameters():
    param.requires_grad = False

# MNIST has 10 digits (0 to 9), so we change the final Fully Connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)
model = model.to(device)

# 4. Configure the Optimizer and the Loss Function
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 5. Training Loop (Only 2 epochs for demonstration purposes)
epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:
            print(f'[Epoch: {epoch + 1}, Batch: {i + 1}] Loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('Fine-Tuning Finished!')

# 6. Save the model weights
model_path = 'mnist_resnet18_weights.pth'
torch.save(model.state_dict(), model_path)
print(f"Model successfully saved to: {model_path}")

Training on device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 459kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.39MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.75MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s]


[Epoch: 1, Batch: 100] Loss: 1.206
[Epoch: 1, Batch: 200] Loss: 0.511
[Epoch: 1, Batch: 300] Loss: 0.395
[Epoch: 1, Batch: 400] Loss: 0.307
[Epoch: 1, Batch: 500] Loss: 0.274
[Epoch: 1, Batch: 600] Loss: 0.255
[Epoch: 1, Batch: 700] Loss: 0.232
[Epoch: 1, Batch: 800] Loss: 0.211
[Epoch: 1, Batch: 900] Loss: 0.199
[Epoch: 2, Batch: 100] Loss: 0.193
[Epoch: 2, Batch: 200] Loss: 0.178
[Epoch: 2, Batch: 300] Loss: 0.168
[Epoch: 2, Batch: 400] Loss: 0.182
[Epoch: 2, Batch: 500] Loss: 0.167
[Epoch: 2, Batch: 600] Loss: 0.165
[Epoch: 2, Batch: 700] Loss: 0.165
[Epoch: 2, Batch: 800] Loss: 0.155
[Epoch: 2, Batch: 900] Loss: 0.160
Fine-Tuning Finished!
Model successfully saved to: mnist_resnet18_weights.pth


In [2]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18
from PIL import Image, ImageOps

# 1. Load the Model and Weights
model = resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)
# Map to CPU, since the free HF Spaces tier does not use GPU
model.load_state_dict(torch.load('mnist_resnet18_weights.pth', map_location=torch.device('cpu')))
model.eval()

# 2. Image Transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 3. Prediction Function
def predict(image_data):
    if image_data is None:
        return None

    # Handle the input difference between Sketchpad (dict) and Image Upload (PIL)
    if isinstance(image_data, dict):
        img = image_data["composite"].convert('L')
        # Invert colors if the sketch background is white and the line is black (MNIST is the opposite)
        img = ImageOps.invert(img)
    else:
        img = image_data.convert('L')

    img_t = transform(img).unsqueeze(0) # Add batch dimension

    with torch.no_grad():
        outputs = model(img_t)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    return {str(i): probabilities[i].item() for i in range(10)}

# 4. Gradio Interface with Tabs
with gr.Blocks() as demo:
    gr.Markdown("# MNIST Digit Classifier")
    gr.Markdown("Draw a number or upload an image to see the estimated probabilities by ResNet18.")

    with gr.Row():
        with gr.Tab("Draw"):
            sketch_input = gr.Sketchpad(label="Virtual Pad (0-9)", type="pil")
            predict_btn_sketch = gr.Button("Predict Drawing")

        with gr.Tab("Upload Image"):
            image_input = gr.Image(sources=["upload", "webcam"], type="pil", label="Upload or Webcam")
            predict_btn_image = gr.Button("Predict Image")

        with gr.Column():
            output_label = gr.Label(num_top_classes=10, label="Estimated Probabilities")

    # Link buttons to the prediction function
    predict_btn_sketch.click(fn=predict, inputs=sketch_input, outputs=output_label)
    predict_btn_image.click(fn=predict, inputs=image_input, outputs=output_label)

    # Add examples to the image section (ensure these images are in the same HF Space folder)
    gr.Examples(examples=["example_1.png", "example_2.png"], inputs=image_input)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a080b2d3b717849668.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
